<a href="https://colab.research.google.com/github/Gael199/Final_Projet/blob/master/Projet_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pyspark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySpark_Student_Exercises_Movies")
    .master("local[*]")
    .getOrCreate()
)

spark

In [2]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q ml-latest-small.zip

ratings = spark.read.csv("ml-latest-small/ratings.csv", header=True, inferSchema=True)
movies  = spark.read.csv("ml-latest-small/movies.csv", header=True, inferSchema=True)

In [3]:
# Full solution for the MovieLens exercises (PySpark)
# Paste into your notebook after the setup cell that creates `spark`
# and after you've loaded `ratings` and `movies` as in your notebook.

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, FloatType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time

In [ ]:
# Task 1 — Data Exploration
# -------------------------

In [4]:
# 1.1 Inspect the data
print("== First 10 rows of ratings ==")
ratings.show(10, truncate=False)

print("== Schema ratings ==")
ratings.printSchema()
print("== Schema movies ==")
movies.printSchema()

print("== Row counts ==")
print("ratings:", ratings.count())
print("movies:", movies.count())

== First 10 rows of ratings ==
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
|1     |47     |5.0   |964983815|
|1     |50     |5.0   |964982931|
|1     |70     |3.0   |964982400|
|1     |101    |5.0   |964980868|
|1     |110    |4.0   |964982176|
|1     |151    |5.0   |964984041|
|1     |157    |5.0   |964984100|
+------+-------+------+---------+
only showing top 10 rows

== Schema ratings ==
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

== Schema movies ==
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

== Row counts ==
ratings: 100836
movies: 9742


In [5]:
# 1.2 Column selection
ratings_sel = (
    ratings
    .select("userId", F.col("movieId").alias("film_id"), F.col("rating").cast(FloatType()).alias("rating"))
)
ratings_sel.show(5)

+------+-------+------+
|userId|film_id|rating|
+------+-------+------+
|     1|      1|   4.0|
|     1|      3|   4.0|
|     1|      6|   4.0|
|     1|     47|   5.0|
|     1|     50|   5.0|
+------+-------+------+
only showing top 5 rows



In [6]:
# 1.3 Filtering
r_user1 = ratings.filter(F.col("userId") == 1)
r_user1.show(10)

r_gt_45 = ratings.filter(F.col("rating") > 4.5)
r_gt_45.show(10)

r_specific = ratings.filter(F.col("movieId").isin([1, 50, 100]))
r_specific.show(10)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|     70|   3.0|964982400|
|     1|    101|   5.0|964980868|
|     1|    110|   4.0|964982176|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
+------+-------+------+---------+
only showing top 10 rows

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|    101|   5.0|964980868|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
|     1|    163|   5.0|964983650|
|     1|    216|   5.0|964981208|
|     1|    231|   5.0|964981179|
|     1|    260|   5.0|964981680|
|     1|    333|   5.0|964981179|
+------+-------+------+---------+
only showing top 10 ro

In [7]:
# 1.4 Sorting & limiting
print("Top 20 highest ratings (ties included):")
ratings.orderBy(F.col("rating").desc(), F.col("timestamp").asc()).show(20)

print("10 lowest ratings by user 600:")
ratings.filter(F.col("userId") == 600).orderBy(F.col("rating").asc(), F.col("timestamp").asc()).show(10)

Top 20 highest ratings (ties included):
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|   429|    150|   5.0|828124615|
|   429|    161|   5.0|828124615|
|   429|    588|   5.0|828124615|
|   429|    590|   5.0|828124615|
|   429|    592|   5.0|828124615|
|   429|    595|   5.0|828124615|
|   429|    168|   5.0|828124616|
|   429|    185|   5.0|828124616|
|   429|    207|   5.0|828124616|
|   429|    252|   5.0|828124616|
|   429|    261|   5.0|828124616|
|   429|    270|   5.0|828124616|
|   429|    289|   5.0|828124616|
|   429|    292|   5.0|828124616|
|   429|    317|   5.0|828124616|
|   429|    339|   5.0|828124616|
|   429|    380|   5.0|828124616|
|   107|      2|   5.0|829322340|
|   107|     11|   5.0|829322340|
|   107|     62|   5.0|829322340|
+------+-------+------+---------+
only showing top 20 rows

10 lowest ratings by user 600:
+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+-

In [8]:
# 1.5 Derived columns
ratings_derived = (
    ratings
    .withColumn("rating_x2", F.col("rating") * 2)
    .withColumn("positive_rating", F.when(F.col("rating") >= 4.0, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("log_rating", F.log10(F.col("rating") + F.lit(1)))
)
ratings_derived.select("userId", "movieId", "rating", "rating_x2", "positive_rating", "log_rating").show(5)

+------+-------+------+---------+---------------+------------------+
|userId|movieId|rating|rating_x2|positive_rating|        log_rating|
+------+-------+------+---------+---------------+------------------+
|     1|      1|   4.0|      8.0|              1|0.6989700043360189|
|     1|      3|   4.0|      8.0|              1|0.6989700043360189|
|     1|      6|   4.0|      8.0|              1|0.6989700043360189|
|     1|     47|   5.0|     10.0|              1|0.7781512503836436|
|     1|     50|   5.0|     10.0|              1|0.7781512503836436|
+------+-------+------+---------+---------------+------------------+
only showing top 5 rows



In [9]:
# 1.6 Missing values (demo)
ratings_null_demo = ratings.withColumn("fake_null", F.when(F.rand() < 0.001, None).otherwise(F.lit("x")))
print("Fake null sample:")
ratings_null_demo.select("fake_null").show(5)

Fake null sample:
+---------+
|fake_null|
+---------+
|        x|
|        x|
|        x|
|        x|
|        x|
+---------+
only showing top 5 rows



In [10]:
# fill
filled = ratings_null_demo.fillna({"fake_null":"missing_string"})
# replace
replaced = ratings_null_demo.na.replace("x", "replaced", subset=["fake_null"])
# drop
dropped = ratings_null_demo.na.drop(subset=["fake_null"])
print("fill / replace / drop examples prepared.")

# 1.7 Distinct & deduplication
unique_users = ratings.select("userId").distinct().count()
unique_movies = ratings.select("movieId").distinct().count()
print("unique users:", unique_users)
print("unique movies rated:", unique_movies)

ratings_dedup = ratings.dropDuplicates(["userId", "movieId"])
print("rows after dedup (if duplicates existed):", ratings_dedup.count())

fill / replace / drop examples prepared.
unique users: 610
unique movies rated: 9724
rows after dedup (if duplicates existed): 100836


In [ ]:
# -------------------------
# Task 2 — Aggregations & GroupBy
# -------------------------

In [11]:
# 2.1 Simple aggregations
agg_stats = ratings.agg(
    F.min("rating").alias("min_rating"),
    F.max("rating").alias("max_rating"),
    F.avg("rating").alias("avg_rating"),
    F.count("*").alias("total_ratings")
)
agg_stats.show()

ratings_per_movie = ratings.groupBy("movieId").count().withColumnRenamed("count", "ratings_count")
ratings_per_movie.show(5)


+----------+----------+-----------------+-------------+
|min_rating|max_rating|       avg_rating|total_ratings|
+----------+----------+-----------------+-------------+
|       0.5|       5.0|3.501556983616962|       100836|
+----------+----------+-----------------+-------------+

+-------+-------------+
|movieId|ratings_count|
+-------+-------------+
|   1580|          165|
|   2366|           25|
|   3175|           75|
|   1088|           42|
|  32460|            4|
+-------+-------------+
only showing top 5 rows



In [12]:
# 2.2 GroupBy more
avg_rating_per_movie = ratings.groupBy("movieId").agg(F.avg("rating").alias("avg_rating"))
count_per_movie = ratings.groupBy("movieId").agg(F.count("rating").alias("count_rating"))

avg_rating_per_user = ratings.groupBy("userId").agg(F.avg("rating").alias("avg_rating_user"))
count_per_user = ratings.groupBy("userId").agg(F.count("rating").alias("count_rating_user"))

In [13]:
# 2.3 Top items
top20_most_rated = count_per_movie.orderBy(F.col("count_rating").desc()).limit(20)
print("Top 20 most-rated movies (movieId, count):")
top20_most_rated.show(20)

Top 20 most-rated movies (movieId, count):
+-------+------------+
|movieId|count_rating|
+-------+------------+
|    356|         329|
|    318|         317|
|    296|         307|
|    593|         279|
|   2571|         278|
|    260|         251|
|    480|         238|
|    110|         237|
|    589|         224|
|    527|         220|
|   2959|         218|
|      1|         215|
|   1196|         211|
|     50|         204|
|   2858|         204|
|     47|         203|
|    780|         202|
|    150|         201|
|   1198|         200|
|   4993|         198|
+-------+------------+



In [14]:
# Top 20 best-rated movies with min 50 ratings
min_ratings_threshold = 50
movie_stats = (
    ratings.groupBy("movieId")
    .agg(F.count("rating").alias("n_ratings"), F.avg("rating").alias("avg_rating"))
)
best_20 = movie_stats.filter(F.col("n_ratings") >= min_ratings_threshold).orderBy(F.col("avg_rating").desc()).limit(20)
print(f"Top 20 best-rated movies (min {min_ratings_threshold} ratings):")
best_20.show(20)

Top 20 best-rated movies (min 50 ratings):
+-------+---------+------------------+
|movieId|n_ratings|        avg_rating|
+-------+---------+------------------+
|    318|      317| 4.429022082018927|
|    858|      192|         4.2890625|
|   2959|      218| 4.272935779816514|
|   1276|       57| 4.271929824561403|
|    750|       97| 4.268041237113402|
|    904|       84| 4.261904761904762|
|   1221|      129|  4.25968992248062|
|  48516|      107| 4.252336448598131|
|   1213|      126|              4.25|
|    912|      100|              4.24|
|  58559|      149| 4.238255033557047|
|     50|      204| 4.237745098039215|
|   1197|      142| 4.232394366197183|
|    260|      251| 4.231075697211155|
|    527|      220|             4.225|
|   1208|      107| 4.219626168224299|
|   2329|      129| 4.217054263565892|
|   1196|      211|4.2156398104265405|
|   1252|       59| 4.211864406779661|
|   1198|      200|            4.2075|
+-------+---------+------------------+



In [15]:
# 2.4 Window functions
w_movie = Window.partitionBy("movieId").orderBy("timestamp")
# rank users by timestamp within each movie (earliest -> latest)
ranked = ratings.withColumn("rank_by_time", F.row_number().over(w_movie))
ranked.select("userId", "movieId", "rating", "timestamp", "rank_by_time").show(10)

# lag: previous rating for the same movie
ratings_with_lag = ratings.withColumn("prev_rating", F.lag("rating").over(w_movie))
ratings_with_lag.select("userId", "movieId", "rating", "prev_rating").show(10)

# average rating per movie with window (will have same avg repeated per row)
w_movie_avg = Window.partitionBy("movieId")
ratings_with_movie_avg = ratings.withColumn("movie_avg", F.avg("rating").over(w_movie_avg))
ratings_with_movie_avg.select("movieId", "rating", "movie_avg").show(10)

+------+-------+------+---------+------------+
|userId|movieId|rating|timestamp|rank_by_time|
+------+-------+------+---------+------------+
|   107|      1|   4.0|829322340|           1|
|   191|      1|   4.0|829759809|           2|
|    54|      1|   3.0|830247330|           3|
|   468|      1|   4.0|831400444|           4|
|   353|      1|   5.0|831939685|           5|
|    40|      1|   5.0|832058959|           6|
|   604|      1|   3.0|832079851|           7|
|   145|      1|   5.0|832105242|           8|
|   130|      1|   3.0|832589610|           9|
|   134|      1|   3.0|832841103|          10|
+------+-------+------+---------+------------+
only showing top 10 rows

+------+-------+------+-----------+
|userId|movieId|rating|prev_rating|
+------+-------+------+-----------+
|   107|      1|   4.0|       NULL|
|   191|      1|   4.0|        4.0|
|    54|      1|   3.0|        4.0|
|   468|      1|   4.0|        3.0|
|   353|      1|   5.0|        4.0|
|    40|      1|   5.0|     

In [16]:
# -------------------------
# Task 3 — Spark SQL
# -------------------------
ratings.createOrReplaceTempView("ratings_table")
movies.createOrReplaceTempView("movies_table")

spark.sql("SELECT * FROM ratings_table LIMIT 20").show(20)
print("Total ratings (SQL):", spark.sql("SELECT COUNT(*) as cnt FROM ratings_table").collect()[0]["cnt"])
print("Distinct users (SQL):", spark.sql("SELECT COUNT(DISTINCT userId) as ucnt FROM ratings_table").collect()[0]["ucnt"])
print("Average rating overall (SQL):", spark.sql("SELECT AVG(rating) as avg_rating FROM ratings_table").collect()[0]["avg_rating"])

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|     70|   3.0|964982400|
|     1|    101|   5.0|964980868|
|     1|    110|   4.0|964982176|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
|     1|    163|   5.0|964983650|
|     1|    216|   5.0|964981208|
|     1|    223|   3.0|964980985|
|     1|    231|   5.0|964981179|
|     1|    235|   4.0|964980908|
|     1|    260|   5.0|964981680|
|     1|    296|   3.0|964982967|
|     1|    316|   3.0|964982310|
|     1|    333|   5.0|964981179|
|     1|    349|   4.0|964982563|
+------+-------+------+---------+

Total ratings (SQL): 100836
Distinct users (SQL): 610
Average rating overall (SQL): 3.501556983616962


In [17]:
# SQL Grouping examples
spark.sql("""
SELECT movieId, AVG(rating) as avg_rating, COUNT(*) as n_ratings
FROM ratings_table
GROUP BY movieId
ORDER BY n_ratings DESC
LIMIT 10
""").show(10)

+-------+-----------------+---------+
|movieId|       avg_rating|n_ratings|
+-------+-----------------+---------+
|    356|4.164133738601824|      329|
|    318|4.429022082018927|      317|
|    296|4.197068403908795|      307|
|    593|4.161290322580645|      279|
|   2571|4.192446043165468|      278|
|    260|4.231075697211155|      251|
|    480|             3.75|      238|
|    110|4.031645569620253|      237|
|    589|3.970982142857143|      224|
|    527|            4.225|      220|
+-------+-----------------+---------+



In [18]:
# Best movies with at least 100 ratings
spark.sql(f"""
SELECT movieId, AVG(rating) as avg_rating, COUNT(*) as n_ratings
FROM ratings_table
GROUP BY movieId
HAVING COUNT(*) >= 100
ORDER BY avg_rating DESC
LIMIT 10
""").show(10)

+-------+-----------------+---------+
|movieId|       avg_rating|n_ratings|
+-------+-----------------+---------+
|    318|4.429022082018927|      317|
|    858|        4.2890625|      192|
|   2959|4.272935779816514|      218|
|   1221| 4.25968992248062|      129|
|  48516|4.252336448598131|      107|
|   1213|             4.25|      126|
|    912|             4.24|      100|
|  58559|4.238255033557047|      149|
|     50|4.237745098039215|      204|
|   1197|4.232394366197183|      142|
+-------+-----------------+---------+



In [19]:
# Worst movies with at least 100 ratings
spark.sql(f"""
SELECT movieId, AVG(rating) as avg_rating, COUNT(*) as n_ratings
FROM ratings_table
GROUP BY movieId
HAVING COUNT(*) >= 100
ORDER BY avg_rating ASC
LIMIT 10
""").show(10)

+-------+------------------+---------+
|movieId|        avg_rating|n_ratings|
+-------+------------------+---------+
|    208|2.9130434782608696|      115|
|    153|2.9160583941605838|      137|
|    586|2.9956896551724137|      116|
|    434|3.0346534653465347|      101|
|    185|3.0401785714285716|      112|
|    344| 3.040372670807453|      161|
|    231|3.0601503759398496|      133|
|   2628| 3.107142857142857|      140|
|    367|3.1847133757961785|      157|
|   2683|3.1983471074380163|      121|
+-------+------------------+---------+



In [20]:
# Most active users
spark.sql("""
SELECT userId, COUNT(*) as n_ratings
FROM ratings_table
GROUP BY userId
ORDER BY n_ratings DESC
LIMIT 10
""").show(10)

+------+---------+
|userId|n_ratings|
+------+---------+
|   414|     2698|
|   599|     2478|
|   474|     2108|
|   448|     1864|
|   274|     1346|
|   610|     1302|
|    68|     1260|
|   380|     1218|
|   606|     1115|
|   288|     1055|
+------+---------+



In [21]:
# 3.3 CASE WHEN rating buckets
spark.sql("""
SELECT rating,
CASE
  WHEN rating >= 4.0 THEN 'high'
  WHEN rating >= 3.0 THEN 'medium'
  ELSE 'low'
END as bucket,
COUNT(*) as cnt
FROM ratings_table
GROUP BY rating, bucket
ORDER BY rating DESC
LIMIT 10
""").show(10)

+------+------+-----+
|rating|bucket|  cnt|
+------+------+-----+
|   5.0|  high|13211|
|   4.5|  high| 8551|
|   4.0|  high|26818|
|   3.5|medium|13136|
|   3.0|medium|20047|
|   2.5|   low| 5550|
|   2.0|   low| 7551|
|   1.5|   low| 1791|
|   1.0|   low| 2811|
|   0.5|   low| 1370|
+------+------+-----+



In [ ]:
# 3.4 SQL Window examples (Top 10 movies by rating for each genre requires join & genre parsing — moved to Task 4)

# -------------------------
# Task 4 — Joins & Genre Analytics
# -------------------------

In [22]:
# 4.1 Basic join
joined = ratings.join(movies, on="movieId", how="inner")
joined.select("userId", "movieId", "title", "rating").show(20, truncate=False)

+------+-------+-----------------------------------------+------+
|userId|movieId|title                                    |rating|
+------+-------+-----------------------------------------+------+
|1     |1      |Toy Story (1995)                         |4.0   |
|1     |3      |Grumpier Old Men (1995)                  |4.0   |
|1     |6      |Heat (1995)                              |4.0   |
|1     |47     |Seven (a.k.a. Se7en) (1995)              |5.0   |
|1     |50     |Usual Suspects, The (1995)               |5.0   |
|1     |70     |From Dusk Till Dawn (1996)               |3.0   |
|1     |101    |Bottle Rocket (1996)                     |5.0   |
|1     |110    |Braveheart (1995)                        |4.0   |
|1     |151    |Rob Roy (1995)                           |5.0   |
|1     |157    |Canadian Bacon (1995)                    |5.0   |
|1     |163    |Desperado (1995)                         |5.0   |
|1     |216    |Billy Madison (1995)                     |5.0   |
|1     |22

In [23]:
# Count how many ratings each genre has: first explode genres
# 4.2 Parse genres
movies_with_genres = movies.withColumn("genres_array", F.split(F.col("genres"), "\\|"))
movies_exploded = movies_with_genres.select("movieId", "title", F.explode("genres_array").alias("genre"))

ratings_movies = ratings.join(movies_exploded, on="movieId", how="inner")
ratings_movies.select("movieId", "title", "genre", "rating").show(10, truncate=False)

+-------+-----------------------+---------+------+
|movieId|title                  |genre    |rating|
+-------+-----------------------+---------+------+
|1      |Toy Story (1995)       |Fantasy  |4.0   |
|1      |Toy Story (1995)       |Comedy   |4.0   |
|1      |Toy Story (1995)       |Children |4.0   |
|1      |Toy Story (1995)       |Animation|4.0   |
|1      |Toy Story (1995)       |Adventure|4.0   |
|3      |Grumpier Old Men (1995)|Romance  |4.0   |
|3      |Grumpier Old Men (1995)|Comedy   |4.0   |
|6      |Heat (1995)            |Thriller |4.0   |
|6      |Heat (1995)            |Crime    |4.0   |
|6      |Heat (1995)            |Action   |4.0   |
+-------+-----------------------+---------+------+
only showing top 10 rows



In [24]:
# count ratings per genre
ratings_per_genre = ratings_movies.groupBy("genre").agg(F.count("*").alias("n_ratings"), F.avg("rating").alias("avg_rating")).orderBy(F.col("n_ratings").desc())
ratings_per_genre.show(20)

+------------------+---------+------------------+
|             genre|n_ratings|        avg_rating|
+------------------+---------+------------------+
|             Drama|    41928|3.6561844113718758|
|            Comedy|    39053|3.3847207640898267|
|            Action|    30635| 3.447984331646809|
|          Thriller|    26452|3.4937055799183425|
|         Adventure|    24161|3.5086089151939075|
|           Romance|    18124|3.5065107040388437|
|            Sci-Fi|    17243| 3.455721162210752|
|             Crime|    16681| 3.658293867274144|
|           Fantasy|    11834|3.4910005070136894|
|          Children|     9208| 3.412956125108601|
|           Mystery|     7674| 3.632460255407871|
|            Horror|     7291| 3.258195034974626|
|         Animation|     6988|3.6299370349170004|
|               War|     4859|   3.8082938876312|
|              IMAX|     4145| 3.618335343787696|
|           Musical|     4138|3.5636781053649105|
|           Western|     1930| 3.583937823834197|


In [25]:
# 4.3 Aggregations: average rating per genre, per movie title, counts
avg_rating_per_genre = ratings_movies.groupBy("genre").agg(F.avg("rating").alias("avg_rating_genre")).orderBy(F.col("avg_rating_genre").desc())
avg_rating_per_genre.show()

avg_rating_per_title = joined.groupBy("title").agg(F.avg("rating").alias("avg_rating"), F.count("*").alias("n_ratings")).orderBy(F.col("avg_rating").desc())
avg_rating_per_title.show(20)

+------------------+------------------+
|             genre|  avg_rating_genre|
+------------------+------------------+
|         Film-Noir| 3.920114942528736|
|               War|   3.8082938876312|
|       Documentary| 3.797785069729286|
|             Crime| 3.658293867274144|
|             Drama|3.6561844113718758|
|           Mystery| 3.632460255407871|
|         Animation|3.6299370349170004|
|              IMAX| 3.618335343787696|
|           Western| 3.583937823834197|
|           Musical|3.5636781053649105|
|         Adventure|3.5086089151939075|
|           Romance|3.5065107040388437|
|          Thriller|3.4937055799183425|
|           Fantasy|3.4910005070136894|
|(no genres listed)|3.4893617021276597|
|            Sci-Fi| 3.455721162210752|
|            Action| 3.447984331646809|
|          Children| 3.412956125108601|
|            Comedy|3.3847207640898267|
|            Horror| 3.258195034974626|
+------------------+------------------+

+--------------------+----------+------

In [26]:
# number of ratings per genre per year (extract year from title if present like "Movie (1995)")
# attempt to extract year e.g. (1995)
movies_with_year = movies_exploded.withColumn("year", F.regexp_extract("title", r".*\\((\\d{4})\\).*", 1))
ratings_movies_year = ratings.join(movies_with_year, on="movieId", how="inner")
ratings_by_genre_year = ratings_movies_year.groupBy("genre", "year").agg(F.count("*").alias("n_ratings")).orderBy("genre", "year")
ratings_by_genre_year.show(20)

+------------------+----+---------+
|             genre|year|n_ratings|
+------------------+----+---------+
|(no genres listed)|    |       47|
|            Action|    |    30635|
|         Adventure|    |    24161|
|         Animation|    |     6988|
|          Children|    |     9208|
|            Comedy|    |    39053|
|             Crime|    |    16681|
|       Documentary|    |     1219|
|             Drama|    |    41928|
|           Fantasy|    |    11834|
|         Film-Noir|    |      870|
|            Horror|    |     7291|
|              IMAX|    |     4145|
|           Musical|    |     4138|
|           Mystery|    |     7674|
|           Romance|    |    18124|
|            Sci-Fi|    |    17243|
|          Thriller|    |    26452|
|               War|    |     4859|
|           Western|    |     1930|
+------------------+----+---------+



In [27]:
# 4.4 Left Anti Join: movies without ratings
movies_no_ratings = movies.join(ratings, on="movieId", how="left_anti")
print("Number of movies with no ratings:", movies_no_ratings.count())
movies_no_ratings.show(20, truncate=False)

Number of movies with no ratings: 18
+-------+--------------------------------------------+------------------------+
|movieId|title                                       |genres                  |
+-------+--------------------------------------------+------------------------+
|1076   |Innocents, The (1961)                       |Drama|Horror|Thriller   |
|2939   |Niagara (1953)                              |Drama|Thriller          |
|3338   |For All Mankind (1989)                      |Documentary             |
|3456   |Color of Paradise, The (Rang-e khoda) (1999)|Drama                   |
|4194   |I Know Where I'm Going! (1945)              |Drama|Romance|War       |
|5721   |Chosen, The (1981)                          |Drama                   |
|6668   |Road Home, The (Wo de fu qin mu qin) (1999) |Drama|Romance           |
|6849   |Scrooge (1970)                              |Drama|Fantasy|Musical   |
|7020   |Proof (1991)                                |Comedy|Drama|Romance    |
|77

In [ ]:
# -------------------------
# Task 5 — MLlib Classification
# -------------------------
# Goal: predict whether rating >= 4.0

In [28]:
# 5.1 Create label
ratings_ml = ratings.withColumn("label", F.when(F.col("rating") >= 4.0, 1.0).otherwise(0.0))

In [29]:
# 5.2 Feature engineering
# base features: rating (we'll include original rating too, though using label is derived from it - typically you'd use other features)
# We'll include timestamp and user-based aggregated feature: number of ratings by that user
user_counts = ratings.groupBy("userId").agg(F.count("*").alias("user_n_ratings"))
ratings_ml = ratings_ml.join(user_counts, on="userId", how="left")

In [30]:
# normalized timestamp: scale timestamp to 0-1 via min-max at DataFrame level
ts_stats = ratings_ml.agg(F.min("timestamp").alias("min_ts"), F.max("timestamp").alias("max_ts")).collect()[0]
min_ts = ts_stats["min_ts"]
max_ts = ts_stats["max_ts"]
range_ts = max_ts - min_ts if max_ts != min_ts else 1
ratings_ml = ratings_ml.withColumn("timestamp_norm", (F.col("timestamp") - F.lit(min_ts)) / F.lit(range_ts))

In [31]:
# optional log timestamp
ratings_ml = ratings_ml.withColumn("log_timestamp", F.log(F.col("timestamp") + F.lit(1)))

# Assemble features: timestamp_norm, user_n_ratings (cast to float), optionally rating
assembler = VectorAssembler(inputCols=["timestamp_norm", "user_n_ratings"], outputCol="rawFeatures")
ratings_ml = assembler.transform(ratings_ml)

# scale features
scaler = StandardScaler(inputCol="rawFeatures", outputCol="features", withMean=True, withStd=True)
scaler_model = scaler.fit(ratings_ml)
ratings_ml = scaler_model.transform(ratings_ml)

In [32]:
# 5.3 Split data
train, test = ratings_ml.randomSplit([0.7, 0.3], seed=42)
print("Train count:", train.count(), "Test count:", test.count())

Train count: 70549 Test count: 30287


In [33]:
# 5.4 Train model: Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train)

print("Logistic Regression coefficients:", lr_model.coefficients)
print("Logistic Regression intercept:", lr_model.intercept)

Logistic Regression coefficients: [-0.021968596309504904,-0.38248592764819905]
Logistic Regression intercept: -0.07928720072624292


In [34]:
# Predictions and ROC AUC
preds_lr = lr_model.transform(test)
bce = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc_lr = bce.evaluate(preds_lr)
print("LR ROC AUC:", auc_lr)

LR ROC AUC: 0.6061311897507139


In [35]:
# 5.5 Evaluate: accuracy, precision, recall, confusion matrix
e_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
e_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
e_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
accuracy = e_acc.evaluate(preds_lr)
precision = e_prec.evaluate(preds_lr)
recall = e_rec.evaluate(preds_lr)
print(f"LR accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}")

LR accuracy=0.5866, precision=0.5937, recall=0.5866


In [36]:
# confusion matrix counts
confusion = preds_lr.groupBy("label", "prediction").count().orderBy("label", "prediction")
print("Confusion matrix (label, prediction, count):")
confusion.show()

Confusion matrix (label, prediction, count):
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 7998|
|  0.0|       1.0| 7778|
|  1.0|       0.0| 4742|
|  1.0|       1.0| 9769|
+-----+----------+-----+



In [37]:
# 5.6 Try DecisionTree and RandomForest (basic comparison)
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label")
dt_model = dt.fit(train)
preds_dt = dt_model.transform(test)
auc_dt = bce.evaluate(preds_dt)
acc_dt = e_acc.evaluate(preds_dt)
print("DT AUC:", auc_dt, "Acc:", acc_dt)

rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50)
rf_model = rf.fit(train)
preds_rf = rf_model.transform(test)
auc_rf = bce.evaluate(preds_rf)
acc_rf = e_acc.evaluate(preds_rf)
print("RF AUC:", auc_rf, "Acc:", acc_rf)

DT AUC: 0.5912869064987141 Acc: 0.6122428764816588
RF AUC: 0.6523963888414788 Acc: 0.6125070162115759


In [38]:
# Print some metrics comparison
print("Model comparison (AUC): LR={:.4f}, DT={:.4f}, RF={:.4f}".format(auc_lr, auc_dt, auc_rf))

Model comparison (AUC): LR=0.6061, DT=0.5913, RF=0.6524


In [ ]:
# -------------------------
# Task 6 — Performance & Execution
# -------------------------

In [39]:
# 6.2 Repartition and coalesce (examples)
ratings_repart = ratings.repartition(8)  # increase partitions (may shuffle)
print("after repartition:", ratings_repart.rdd.getNumPartitions())

ratings_coalesced = ratings_repart.coalesce(2)  # decrease partitions (may avoid heavy shuffle)
print("after coalesce:", ratings_coalesced.rdd.getNumPartitions())

after repartition: 8
after coalesce: 2


In [40]:
# 6.3 Cache
ratings.cache()
t0 = time.time()
cnt = ratings.count()  # triggers caching
t1 = time.time()
print("First count (cache fill) time:", t1 - t0)

t0 = time.time()
avg1 = ratings.groupBy("movieId").agg(F.avg("rating").alias("avg")).count()
t1 = time.time()
print("First groupBy time:", t1 - t0)

t0 = time.time()
avg2 = ratings.groupBy("movieId").agg(F.avg("rating").alias("avg")).count()
t1 = time.time()
print("Second groupBy time (should be faster with cache):", t1 - t0)

First count (cache fill) time: 1.064145803451538
First groupBy time: 0.39792346954345703
Second groupBy time (should be faster with cache): 0.26094532012939453


In [41]:
# 6.4 explain(True) — examples
print("Explain join:")
joined.explain(True)

print("Explain groupBy:")
ratings.groupBy("movieId").agg(F.avg("rating").alias("avg_rating")).explain(True)

print("Explain window (example):")
ratings_with_movie_avg.select("movieId", "rating", "movie_avg").explain(True)

Explain join:
== Parsed Logical Plan ==
'Join UsingJoin(Inner, [movieId])
:- Relation [userId#17,movieId#18,rating#19,timestamp#20] csv
+- Relation [movieId#42,title#43,genres#44] csv

== Analyzed Logical Plan ==
movieId: int, userId: int, rating: double, timestamp: int, title: string, genres: string
Project [movieId#18, userId#17, rating#19, timestamp#20, title#43, genres#44]
+- Join Inner, (movieId#18 = movieId#42)
   :- Relation [userId#17,movieId#18,rating#19,timestamp#20] csv
   +- Relation [movieId#42,title#43,genres#44] csv

== Optimized Logical Plan ==
Project [movieId#18, userId#17, rating#19, timestamp#20, title#43, genres#44]
+- Join Inner, (movieId#18 = movieId#42)
   :- Filter isnotnull(movieId#18)
   :  +- InMemoryRelation [userId#17, movieId#18, rating#19, timestamp#20], StorageLevel(disk, memory, deserialized, 1 replicas)
   :        +- FileScan csv [userId#17,movieId#18,rating#19,timestamp#20] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 

In [ ]:










# -------------------------
# Stop Spark (if you want)
# -------------------------
# spark.stop()  # uncomment if you want to stop the session here
